[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eth-fdd-fs26/FDD-WE7-public/blob/main/project/notebook/we7_project_federated_learning_student.ipynb)

# One Model, Four Banks

Four banks want to answer the same question: when we lend someone money, will they pay it
back?

Each has a book of past loans and knows how they turned out, so each could build a model from
its own book. None may show that book to the other three.

That is a problem of scale, though not the one the phrase usually brings to mind. Scale is
normally a wall built of hardware: the model will not fit, or one machine is too slow. This
wall is not. Nothing here is too big to compute, and no hardware moves it, because the thing
that cannot move is the data.

So the training goes to the data instead of the data going to the training. That comes with
its own bills: what crosses the network, what each participant has to compute, and what the
whole arrangement gives away.

This notebook works out what these four banks can do about it. Whether working together is
worth anything, what it costs, what it gives away, and whether you would sign it off.

**How to work through it.** Run every cell in order. Six cells have blanks marked `???`, five of
them required and the sixth in the optional part 7, each flagged by a `🎯 Task` comment on the
line above it. Fill them in and run the cell: it marks
your answer and tells you which line to look at if a blank is wrong. Three more cells open a
playground you play rather than read.

**What you hand in** is this notebook, run, with two kinds of thing filled in by you: the `???`
blanks in the five required task cells, and every ⬜ in a table headed **✍️ Fill this in**.
There are four of those tables. A few cells also ask you to predict something before you run
it, which is worth doing honestly: the point of those is the gap between what you expected and
what happened.

In [ ]:
#@title 📥 0.1 — fetch the exercise files (run me) { display-mode: "form" }
import os, subprocess, sys

REPO_OWNER, REPO_NAME = "eth-fdd-fs26", "FDD-WE7-public"
FILES = ["fedcore.py", "fedviz.py", "generator.py", "quizzes.py", "scenario.py"]
RAW = f"https://raw.githubusercontent.com/{REPO_OWNER}/{REPO_NAME}/main/project"


def _in_colab():
    try:
        import google.colab                     # noqa: F401
        return True
    except Exception:
        return False


def _git(*args):
    # subprocess rather than the ! shell magic, so this cell is ordinary Python and the
    # token below never reaches a shell that would log or re-split it
    return subprocess.run(["git", *args], capture_output=True, text=True).returncode == 0


if _in_colab():
    token = ""
    try:                                        # private repo, while we are testing:
        from google.colab import userdata       # read a GITHUB_TOKEN from Colab Secrets
        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:                           # public repo: no token needed
        token = ""
    auth = f"{token}@" if token else ""
    url = f"https://{auth}github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if os.path.isdir(REPO_NAME):
        print("updating the exercise repo ...")
        if not _git("-C", REPO_NAME, "pull", "-q", url):
            print("  (could not pull, so using the copy already here)")
    else:
        print("cloning the exercise repo ...")
        _git("clone", "-q", url)


def _find_src(*roots):
    # Search for the folder rather than naming a path, so the repo can be laid out
    # however it likes without breaking this cell.
    for root in roots:
        if not os.path.isdir(root):
            continue
        for base, _dirs, files in os.walk(root):
            if os.path.basename(base) == "src" and "fedviz.py" in files:
                return os.path.abspath(base)
    return None


SRC = _find_src(REPO_NAME, ".", "..", "../..")
if SRC is None:                                 # no git: fetch the five files directly
    try:
        from urllib.request import urlretrieve
        SRC = os.path.abspath("src")
        os.makedirs(SRC, exist_ok=True)
        for f in FILES:
            urlretrieve(f"{RAW}/src/{f}", os.path.join(SRC, f))
        print("fetched the exercise files directly")
    except Exception:
        raise FileNotFoundError(
            "Could not find the exercise files. If the repo is still private, add a "
            "GITHUB_TOKEN secret in Colab and run this cell again.")

os.chdir(os.path.dirname(SRC))
sys.path.insert(0, SRC)

# Three parts of this notebook end in a playground mission. The pages are about a
# megabyte each, so they are fetched when you reach them rather than now.
os.environ.setdefault("FDD_MISSIONS", f"{RAW}/missions")

print("files ready ✓   working directory:", os.getcwd())

In [ ]:
#@title 📦 0.2 — set up { display-mode: "form" }
import numpy as np
import fedcore as fc
import fedviz as fv
import scenario

fv.use(scenario)
books, cohort = scenario.split()
print("setup complete ✓")

In [ ]:
#@title 🧱 0.3 — which wall this is { display-mode: "form" }
fv.scale_walls(books)

## Meet the four banks

They are not the same kind of business, and it shows in who walks through the door.

**A Prime Metro** is a large city lender with wealthy, settled customers. **B Digital Growth**
lends online to younger people with thin credit files. **C Regional Mainstreet** serves a lower
income region with many self employed customers. **D Community High-Touch** is small and lends
to people the other three would turn away.

The next cell puts all 2,450 customers on one map, one panel per bank, and the one after that
gives the same four books as numbers. Three things to look for on the map:

| what to look for | what it means |
|:--|:--|
| **the four banks sit in different places** · A Prime's customers spread right into higher incomes, D Community's are squeezed into the bottom left corner | these are not four samples of the same crowd |
| **red sits in the same corner of every panel** · low income and high debt means more defaults at all four | the rule connecting a customer to their outcome does not appear to change from bank to bank |
| **nobody sees the whole map** · D Community has 150 customers in one small region | it still has to decide about everyone who walks in |

If the rule is the same everywhere, a bank seeing the whole map would predict better than one
seeing a corner. Part 1 measures whether that holds.

In [ ]:
#@title 🗺️ 0.4 — who each bank lends to { display-mode: "form" }
fv.market_map(scenario.load_raw(), "annual_income_k", "debt_to_income",
              scenario.BUSINESS, scenario.READABLE_BUSINESS,
              x_label="annual income (thousands)", y_label="debt-to-income ratio")

In [ ]:
#@title 📋 0.5 — the four books, in numbers { display-mode: "form" }
print(f"{'bank':<26}{'customers':>10}{'defaults':>10}{'default rate':>14}")
for b in books:
    print(f"{b.name:<26}{b.n:>10}{int(b.y.sum()):>10}{b.share_positive:>13.1%}")
X, y = books.pooled()
print(f"{'TOTAL':<26}{len(y):>10}{int(y.sum()):>10}{y.mean():>13.1%}")

### The question, precisely

For each customer the banks record thirteen things they already have on file, and one outcome:
did this loan default within twelve months.

One of the thirteen is the interest rate the bank charged, which fixes what this model is for.
It predicts the outcome of a loan already priced and granted, the question a risk team asks
about a portfolio it holds. It is not an approval model, which would have to drop the rate,
because at approval nobody has set one yet.

In [ ]:
#@title 🧮 0.6 — the model we will use all the way through { display-mode: "form" }
_ill = fc.train(fc.init(), *books.pooled(), 3000, lr=0.5)   # trained here just to show
_who = books[3]
fv.model_card(_who.X[0], _ill[:fc.N_FEATURES], scenario.FEATURES, scenario.READABLE,
              bias=_ill[fc.N_FEATURES], outcome=_who.y[0],
              filed=scenario.filed(_who.X[0]))

### Reading the scorecard

Fourteen numbers: one for each thing on file, plus a starting number. That is the entire
model, and the figure shows only the three that move this customer most.

The card also shows this customer as their bank filed them. The model never sees those units:
every column is shifted and scaled onto a common footing first, identically at all four banks.
That table of shifts and scales is the consortium's **dictionary**, and part 5 comes back to it.

| on the card | what it does |
|:--|:--|
| debt-to-income and credit utilisation, both well above typical here | push the score up |
| no late payments | pushes it down |
| the starting number, −2.57 | most customers do not default, so the model begins by assuming they will not |

The curve is what makes it a chance rather than a score: any total lands between 0% and 100%,
and 0 would be exactly 50%.

Training means choosing those fourteen numbers so the predictions match what happened.
**In part 1 you will train them yourself**, and the rest of the notebook is how four banks
agree on them without sharing a record. The model stays this simple throughout, because four
organisations training one model is easier to see when the model is not the complicated part.
The optional part 7 makes it bigger.

### Where the data comes from

The four banks are invented and their customers generated rather than collected, for two
reasons.

No public dataset has four real banks lending to genuinely different populations with the
outcomes attached. And a generated dataset means we know the true answer: every customer's
probability of default comes from one equation, used at all four banks. Part 2 uses that to
check each bank's model against the rule that produced the data.

### What you are going to do

| Parts | Question |
|---|---|
| 1 | Is any one bank's model good enough for all four? |
| 2, 3 | Why do the four disagree, and can they train one model without sharing records? |
| 4 | What does that cost to run, and when does it break? |
| 5, 6 | What does a shared model give away, and what actually protects a customer? |
| 7 *(optional)* | What changes at real model sizes? |

Each part ends by saying where the consortium stands. Part 6 adds the fourth budget, what the
arrangement spends on privacy, and the required notebook ends there.

**What is optional, and what it adds.** Part 7 prices the same procedure at real model and
federation sizes, and two short appendices derive the update and point at further reading.
Nothing required depends on any of it, so you can stop after part 6 and still hand in a complete
notebook. Optional cells are marked in their heading.

**Three of those parts end in a mission.** Parts 3, 5 and 6 finish with a playground that opens
inside this notebook, on these same four banks, where you run the thing the part just explained
instead of reading about it. They are the only cells you play rather than execute.

---
# Part 1 · Train alone, or train together?

The question is narrower than it first sounds. It is not whether a bank's own model is good
for its own customers. It is whether any single bank's model is good enough for all four.

Answering it needs two reference points.

| | What it is | Can the consortium build it? |
|---|---|---|
| **Pooled** | One model trained on all 2,450 records at once, as if the four books were a single database. | **No.** The records cannot be pooled. We build it here only to see the best any method could reach. |
| **Local** | Four models, each trained inside one bank on that bank's own records. | **Yes.** This is what they have today. |

Everything else in this notebook sits between those two points. Part 3 is where a method
appears that reaches the first without ever doing the thing the first requires.

### 🎯 Task 1 · One training step

The model is a logistic regression with fourteen numbers in it: one weight for each of the
thirteen features, plus one more called the bias that shifts every prediction up or down.

Training means starting from all zeros and repeatedly nudging those fourteen numbers in the
direction that makes the model's mistakes smaller. That direction is called the **gradient**.
You compute it from the data, then take a small step against it. Repeat a few thousand times
and you have a trained model.

Fill in the two blanks below. `fc.gradient(w, X, y)` gives you the direction for the current
model `w` on the records `X, y`. The step size `lr` says how far to move each time.

In [ ]:
def fit_local(X, y, steps=3000, lr=0.5):
    """Train the model on one bank's records."""
    w = fc.init()                       # fourteen numbers, all zero
    for _ in range(steps):
        # 🎯 Task 1a — which way do the mistakes get smaller? fc has a function for it
        g = ???
        # 🎯 Task 1b — a gradient points towards MORE mistakes, so step against it,
        #              by lr times its size
        w = ???
    return w


_w = fit_local(books[0].X, books[0].y)
print(f"loss before training  {fc.loss(fc.init(), books[0].X, books[0].y):.3f}")
print(f"loss after training   {fc.loss(_w, books[0].X, books[0].y):.3f}")

_g    = fc.gradient(fc.init(), books[0].X, books[0].y)
_step = fit_local(books[0].X, books[0].y, steps=1) - fc.init()
_down = np.allclose(_step / (np.linalg.norm(_step) + 1e-12),
                    -_g / np.linalg.norm(_g), atol=1e-6)
fv.check(
    ("Task 1a", _down,
     "the step points straight down the gradient",
     "the step does not follow fc.gradient(w, X, y) — check what g is"),
    ("Task 1b", _down and np.allclose(_step, -0.5 * _g),
     "and its size is exactly lr times the gradient",
     "the direction is right but the size is not lr x g — write w - lr * g"),
)

In [ ]:
local = {b.name: fit_local(b.X, b.y) for b in books}
Xp, yp = books.pooled()
pooled = fit_local(Xp, yp)
print(f"{len(local)} local models trained, plus the pooled reference")

In [ ]:
#@title ⚖️ 1.1 — how good is the pooled model?
Xc, yc = cohort.pooled()
never_defaults = 1.0 - yc.mean()
model_accuracy = fc.accuracy(pooled, Xc, yc)
print(f"a model that never predicts default   {never_defaults:.1%} accurate")
print(f"the pooled model                      {model_accuracy:.1%} accurate")
print(f"difference                            {model_accuracy - never_defaults:+.1%}")

### Would you deploy on that evidence?

Answer before reading on.

The model bought 1.7 percentage points over a rule that ignores every feature and always says
"will not default". That is not a bad model. It is a bad measurement. Only 11.9 per cent of
these customers default, so predicting the common answer is right about almost everyone.
Accuracy here mostly tells you how rare default is.

From here the notebook uses **AUC**. Take one customer who defaulted and one who did not, and
ask how often the model gives the defaulter the higher risk score. 0.5 is guessing, 1.0 never
gets that pair the wrong way round.

The cell below scores all five models that way, each one on every bank's customers.

In [ ]:
#@title 🔍 1.2 — every model, scored on every bank's customers { display-mode: "form" }
models = dict(local)
models["pooled (not allowed)"] = pooled
fv.cross_grid(models, cohort, fc.auc, metric="AUC",
              title=scenario.FIGURES["crossgrid_title"])

### Reading the grid

Each row is a model. Each column is the group of customers it was scored on. The black
outlines mark a bank scoring its own customers.

Two things stand out. First, the four local models sit within about half a point of each
other everywhere. Nobody's model is badly wrong at anybody's bank, which makes sense: the
banks are learning the same rule from different samples of it.

Second, and less comfortable, in three of the four columns some *other* bank's model scores
better on your customers than your own model does. A Prime Metro's model is the best model
for D Community's customers. The bottom row, the pooled model, wins every single column.

In [ ]:
#@title 📈 1.3 — what is each bank losing by working alone? { display-mode: "form" }
gains = {}
for b, c in zip(books, cohort):
    alone = fc.auc(local[b.name], c.X, c.y)
    with_pooling = fc.auc(pooled, c.X, c.y)
    gains[b.name] = with_pooling - alone
    print(f"{b.name:<26} alone {alone:.3f}   with pooling {with_pooling:.3f}"
          f"   gain {with_pooling - alone:+.3f}")

### Every bank is leaving something on the table

Each bank was scored on its own customers, so this is not an average across the consortium.
It is what each member individually would gain.

All four gain, and D Community High-Touch gains the most. That is the smallest bank, with 150
customers.

One run on one draw of the data is thin evidence for a claim about which bank benefits most,
because the order could easily be an accident of who happened to walk into which branch. The
next cell repeats the whole thing on ten independent draws.

In [ ]:
#@title 📊 1.4 — what each bank gains by pooling { display-mode: "form" }
import generator as gen
per_draw = {b.name: [] for b in books}
for s in range(10):
    bk = scenario.load(seed=gen.BOOK_SEED + s * 101)
    wp = fit_local(*bk.pooled())
    for site, c in zip(bk, cohort):
        wl = fit_local(site.X, site.y)
        per_draw[site.name].append(fc.auc(wp, c.X, c.y) - fc.auc(wl, c.X, c.y))

fv.part1_close(per_draw, metric="AUC")

### Which gains survive a rebuild

D Community gains on all ten draws. The other three gain on nine out of ten. Had we run this
once and shown you the mean, you would not know that a single unlucky draw wipes out the gain
for three of the four banks. It cannot for D Community, which is the strongest form the claim
can take on this evidence.

Read the two numbers beside each bank together. The average says how much pooling is worth;
how reliably says whether that average is a number you could put in a paper. A gain that survives
every rebuild and a gain that survives nine out of ten are different claims, and only the
first one belongs to the smallest bank.

**Where part 1 leaves us.** No single bank's model serves the consortium. Pooling would help
all four and would help the smallest most. Pooling is not allowed.

That is the problem the rest of the notebook solves. Before solving it, it is worth knowing
why the four banks disagree in the first place, because the answer decides how hard the
solution has to work.

---
# Part 2 · Do the banks see the same customers?

Part 1 showed four models that disagree. There are two very different reasons that could
happen, and they lead to different amounts of trouble.

| | Why the models differ | How much trouble |
|---|---|---|
| **Less data** | Each bank simply has fewer records than the four of them have together. They are looking at the same kind of customer, just fewer of them. | Little. Combining them is close to free. |
| **Different customers** | Each bank lends to a different sort of person, so each is learning about a different slice of the world. | More. Every method later in this notebook has to cope with it. |

The second one has a name. Data spread across organisations is almost never **identically
distributed**, which is the phrase you will meet in any paper on this subject. It means what
part 0's map already showed: A Prime's customers and D Community's customers are not draws
from the same population, and no amount of arithmetic makes them so.

That is the normal case, not the awkward exception. This part works out which of the two the
four banks are.

In [ ]:
#@title 🎯 2.1 — what each bank worked out on its own { display-mode: "form" }
truth = gen.TRUE_BETA
fits = {b.name: fit_local(b.X, b.y)[:fc.N_FEATURES] for b in books}
fv.rule_recovery(fits, truth, scenario.FEATURES, scenario.READABLE,
                 sites={b.name: b.n for b in books})

### They are all solving the same problem

This is the most important picture in part 2. Each bank fitted the model on its own records, with
no knowledge of the others. The black ticks are the numbers the data was really generated with,
and the coloured bars are what each bank came up with.

| what all four made large and positive | what all four made negative |
|:--|:--|
| debt-to-income, credit utilisation, late payments | income, credit history |

Nobody concluded something different in kind. They are four rough drafts of the same rule, and
where they differ is precision: D Community's bars wobble around the ticks much more than
A Prime's, and one or two are badly off. That is what having less to learn from looks like.

**This is why one shared model is the right goal.** If the four banks genuinely disagreed about
what makes a borrower risky, averaging their models would produce a compromise that suited
nobody. They do not disagree. They are each seeing a blurred version of the same thing, and
blurred versions of the same thing can usefully be combined. The cell below puts a number on how
blurred each one is, beside each book's size and the defaults it has actually seen.

In [ ]:
#@title 📐 2.2 — how close each one got { display-mode: "form" }
print(f"{'bank':<26}{'customers':>11}{'defaults seen':>15}{'agreement with truth':>22}")
for b in books:
    v = fits[b.name]
    agree = float(v @ truth / (np.linalg.norm(v) * np.linalg.norm(truth)))
    print(f"{b.name:<26}{b.n:>11,}{int(b.y.sum()):>15}{agree:>22.3f}")

**Where part 2 leaves us.** Collaboration is worth having and should not be hard, because
these four banks differ in customers rather than in what a default means.

Nobody has yet said how to do it without moving a single record.

---
# Part 3 · Training together without sharing data

Here is the idea the whole field is built on. A bank cannot send its records, but it can send
the *direction* those records would move the model in: the gradient from Task 1, fourteen
numbers, one per parameter.

This part answers two questions in order.

1. **How do four directions become one, and how close does that get to pooling?** Sections 3.1
   to 3.4.
2. **Then you build the loop yourself**, in the playground. Section 3.5.

The figure below is one complete round built on that idea. Follow the numbers, and notice that
it ends on a question rather than an answer.

In [ ]:
#@title 🔄 3.1 — one federated round, step by step { display-mode: "form" }
fv.fedsgd_round(books)

## 1 · How four directions become one

### The obvious way to combine them

The coordinator has four directions and needs one. The obvious move is to average them, giving
each bank an equal say.

Here is a case where that fails, small enough to check on paper. Two banks, four customers
between them.

| | customers | each customer's direction | what the bank sends |
|---|---:|---:|---:|
| **Bank A** | 3 | +2 | +2 |
| **Bank B** | 1 | −2 | −2 |

Pooling all four customers averages +2, +2, +2 and −2, which comes to **+1**. That is the answer
the federation is trying to match, because pooling is what it would do if records could travel.

Averaging the two banks equally gives **0**: one customer at Bank B has cancelled all three at
Bank A. Weighting Bank A by 3/4 and Bank B by 1/4 gives 1.5 − 0.5, which is **+1** again.

In [ ]:
#@title 🧮 3.2 — four customers, two banks, three answers
bank_A = [2.0, 2.0, 2.0]              # three customers, each pointing the same way
bank_B = [-2.0]                       # one customer, pointing the other way

sends_A = np.mean(bank_A)             # a bank sends the average over its own customers
sends_B = np.mean(bank_B)

print(f"pooling all four customers   {np.mean(bank_A + bank_B):+.1f}")
print(f"one vote per bank            {(sends_A + sends_B) / 2:+.1f}")
print(f"one vote per customer        {0.75 * sends_A + 0.25 * sends_B:+.1f}")

### Where those weights come from

Bank A's three quarters was not a guess. It is its share of the customers, and the same
argument works for any number of banks of any size. Five symbols, then two lines of algebra.

| symbol | what it stands for |
|:--|:--|
| $N$ | customers in the whole consortium, here 2,450 |
| $n_k$ | customers at bank $k$ |
| $g_i$ | the direction from customer $i$ on their own |
| $g_k$ | the average direction at bank $k$ |
| $g$ | the direction pooled training would take |

Those directions are what everyone else calls gradients. *Direction* is what they do: each one
points the way the model should move next.

Pooled training averages over every customer. Split that sum up by the bank holding each
customer, and the $n_k$ customers at bank $k$ contribute $n_k g_k$ between them:

$$g \;=\; \frac{1}{N}\sum_{i=1}^{N} g_i
      \;=\; \frac{1}{N}\sum_{k=1}^{4} n_k \, g_k
      \;=\; \sum_{k=1}^{4} \frac{n_k}{N} \, g_k$$

In words: the pooled direction is the four bank directions added together, each counting in
proportion to how many customers its bank holds. In the example above $N$ was 4 and $n_A$ was 3,
which is where the three quarters came from.

### 🎯 Task 2 · Combining the four directions

Now write that rule for the real four banks.

`updates` is a list of four directions. `sizes` is a list of the four customer counts.

In [ ]:
def aggregate(updates, sizes):
    """Combine four bank directions into one."""
    sizes = np.asarray(sizes, float)
    # 🎯 Task 2a — each bank's share of the 2,450 customers, so the four add up to 1
    weights = ???
    # the directions added up with those weights: one number per bank against fourteen
    # per update, so the weights take an extra axis to line up
    return np.sum(weights[:, None] * np.asarray(updates), axis=0)


_sizes = books.sizes
_ups = [fc.gradient(fc.init(), b.X, b.y) for b in books]
_combined = aggregate(_ups, _sizes)
_pooled_direction = fc.gradient(fc.init(), *books.pooled())
print(f"largest difference from the pooled direction: "
      f"{np.abs(_combined - _pooled_direction).max():.2e}")

_shares = np.allclose(aggregate([[0.0], [4.0]], [3, 1]), 1.0)
fv.check(
    ("Task 2a", _shares,
     "three customers against one gives the big bank three quarters of the say",
     "a 3-to-1 split does not give 3/4 — weights must be sizes / sizes.sum()"),
)

In [ ]:
#@title ⚖️ 3.3 — the same test on the real four banks { display-mode: "form" }
equal = np.mean(np.asarray(_ups), axis=0)
print(f"{'bank':<26}{'customers':>11}{'weight':>9}")
for b, wgt in zip(books, _sizes / _sizes.sum()):
    print(f"{b.name:<26}{b.n:>11,}{wgt:>9.3f}")
print()
print(f"weighted by customers   {np.abs(_combined - _pooled_direction).max():.2e}")
print(f"every bank equal        {np.abs(equal - _pooled_direction).max():.3f}")

**Why the next figure is drawn on loss and not AUC.** AUC asks whether the model puts risky
customers above safe ones. The loss asks whether the probability it attaches to each customer
is right. A bank that approves every applicant below a fixed probability cares about the
second, and the second is the one still moving after the first has settled.

In [ ]:
#@title 📉 3.4 — twenty exchanges, against pooled training step for step { display-mode: "form" }
history = fc.run(list(books), E=1, rounds=20, lr=0.5, evaluate_on=cohort)

# the same twenty steps taken centrally, on all 2,450 records at once
w, pooled_curve = fc.init(), []
for _ in range(20):
    w = w - 0.5 * fc.gradient(w, Xp, yp)
    pooled_curve.append(fc.loss(w, Xc, yc))


pooled_loss = fc.loss(fit_local(Xp, yp, steps=20000), Xc, yc)
fv.convergence_overlay(history["loss"], pooled_curve, converged=pooled_loss)

### This is not an approximation

You proved one step. The figure shows twenty. Here is why proving one is enough.

Both sides start from the same model, fourteen zeros. Your task showed the combined direction
is the pooled direction to the last decimal, so step one lands both in the same place. Step two
then starts from the same model, and the argument repeats for as long as you run it.

So federated training does not get *close* to pooled training. It **is** pooled training
computed in four places, which is why the figure shows one curve hidden under another.

The faint rule lower down is where pooled training ends up if you let it finish. Twenty steps
does not reach it, and nor would twenty steps of pooling.

### What the equality depends on

Seven things have to hold, and the notebook breaks them one at a time from here.

| Condition | In plain words | What changes it |
|---|---|---|
| **Everyone takes part** | All four banks join every round | The optional part 7, where only a sample of a large federation speaks each round |
| **Every record is used** | Each bank's step uses its whole book | Part 5 shrinks a step to one customer; part 6 samples |
| **A shared starting point** | Every round begins from the same model at all four | Holds throughout |
| **A shared objective** | Same preprocessing, same loss, everywhere | Holds throughout |
| **Weighted by size** | Each bank counts in proportion to its customers | You measured the alternative: 0.092 away from pooled |
| **One local step** | Exactly one gradient step between exchanges | **Part 4 drops this on purpose** |
| **The coordinator waits** | It combines only once all four have replied | Holds throughout |

Drop any one and the equality becomes an approximation. Whether it is a good one is what the
rest of the notebook keeps asking.

## 2 · Now build it yourself

The next cell opens a playground **inside this notebook**. Same four banks, same 2,450
customers, same arithmetic you have just written, on a board you watch instead of numbers you
read.

**Build the machine.** Put the five steps of a federated round into the right order, then run
the finished loop once and watch the shared score move off 0.500.

Three things before you start.

- It runs in your browser, on records that are already here. Nothing is uploaded anywhere.
- The mission scrolls **inside its own frame**. Use that scrollbar, not the notebook's.
- When the card says **Mission achieved**, press **Back to the notebook** and carry on below.

In [ ]:
#@title 🎮 3.5 — mission: build the machine (run me, then play) { display-mode: "form" }
fv.mission("build")

### ✍️ Fill this in — your build log

**You hand this in.** One line in each ⬜, from what you just saw. Do it before reading on: the
next cell gives part of it away.

| What to record | Your answer |
|---|---|
| What travels from the banks back to the coordinator | ⬜ |
| The shared score after one round, starting from 0.500 | ⬜ |

That loop is the one this part derived. The model goes out, each bank trains on records that
never move, the changes come back, the coordinator averages them in proportion to size. You
wrote the aggregation in Task 2 and have now watched it run.

**Where part 3 leaves us.** The consortium can have pooling's quality without pooling anything,
which is the problem part 0 set out. Exactly, as long as it pays for an exchange after every
single step.

That is the price, and it is a schedule rather than a bill: every step, all four banks online,
computing and replying before anyone continues, and every one of those a chance for somebody to
be late. The model is right and the schedule is not, and that is what part 4 is for.

---
# Part 4 · Train more locally, communicate less

FedSGD solved the consortium's problem: the model is as good as pooling and no record moved.
It solved it by talking after every single step.

The network bill for that is nothing: a fourteen-number model makes tiny messages however many
of them there are. What costs is the schedule. Section 4.1 counts how many of those
conversations one local step per exchange actually needs, and 4.2 puts the same four runs on
both meters at once.

FedAvg asks whether the banks can work longer between conversations and have fewer of them.
This part prices both sides of that trade: what the saving is, and what it is paid for with.

### 🎯 Task 3 · What each scheme costs

Two budgets, and they move in opposite directions.

**Communication** is what crosses the network. Each exchange the coordinator sends the model
down to every bank and every bank sends its update back up, so with $R$ exchanges, $K$ banks and
a model of $S$ bytes that is $R \cdot K \cdot 2S$. Model payload only: handshakes, encryption
and retries are real and are not counted here.

**Computation** is what the banks do. With $E$ local steps per exchange each bank takes
$R \cdot E$ steps. That is per bank, not the consortium total, because all four work at the same
time. It is what the slowest bank grinds through, which is what sets the length of a round.

Fill in both.

In [ ]:
K = len(books)
S = fc.N_PARAMS * 8          # fourteen numbers, eight bytes each

def bytes_moved(rounds, K=K, S=S):
    # 🎯 Task 3a — price one exchange first: the model down to all K banks and an update
    #              back from each, then scale by rounds. One multiplication.
    return ???

def local_steps(rounds, E):
    # 🎯 Task 3b — the same way, for one bank's work: E steps inside each exchange
    return ???


for R, E in [(56, 1), (13, 5), (4, 20), (3, 50)]:
    step = "step " if E == 1 else "steps"
    print(f"{E:>3} local {step} per exchange   {R:>3} exchanges   "
          f"{bytes_moved(R):>7,} bytes   {local_steps(R, E):>5} steps of local work")

fv.check(
    ("Task 3a", bytes_moved(1) == 2 * K * S and bytes_moved(10) == 10 * bytes_moved(1),
     f"one exchange moves {2 * K * S:,} bytes, and ten of them move ten times that",
     f"one exchange should move 2 x {K} banks x {S} bytes = {2 * K * S:,}; "
     "count both directions"),
    ("Task 3b", local_steps(13, 5) == 65 and local_steps(4, 20) == 80,
     "thirteen exchanges of five steps is sixty five steps a bank",
     "13 exchanges at 5 local steps should be 65 steps, not "
     f"{local_steps(13, 5)}"),
)

In [ ]:
#@title ⏱️ 4.1 — how few exchanges can we get away with? { display-mode: "form" }
to_target = {}
print(f"{'local steps':>12}{'exchanges':>11}{'steps per bank':>16}")
for E in (1, 5, 20, 50):
    to_target[E] = fc.exchanges_to_target(books, cohort, E=E, lr=0.5, cap=400)
    print(f"{E:>12}{to_target[E]:>11}{to_target[E] * E:>16}")
print()
print("exchanges: to get within one per cent of where pooled training finishes")

In [ ]:
#@title 📉 4.2 — the same four runs, on two meters { display-mode: "form" }
curves = {}
for E in (1, 5, 20, 50):
    R = to_target[E] + max(2, to_target[E] // 10)      # just past the crossing
    h = fc.run(list(books), E=E, rounds=R, lr=0.5, evaluate_on=cohort)
    ex = np.arange(1, R + 1)
    label = "1 local step" if E == 1 else f"{E} local steps"
    curves[label] = (ex, ex * E, np.array(h["loss"]))
fv.two_panel(curves, pooled_loss)

### Reading the rings

The rings mark where each run first passes the stopping target, and the same four rings appear
in both panels because they are the same four moments.

On the left meter the counts fall as local work rises: 56 exchanges, then 13, 4 and 3. On the
right meter they climb: 56 steps per bank, then 65, 80 and 150. Every setting reaches the same
target. What changes is which meter it runs up, and the two meters move in opposite directions.

Which one you would rather feed depends on your consortium. Four banks on good networks would
take the computing. Four thousand phones would not. A third budget, **privacy**, is how much a
customer is exposed by what the federation sends. It arrives in part 6, and no setting is free
on all three.

**Where part 4 leaves us.** For these four banks, five local steps per exchange reaches the
same one per cent target in thirteen conversations instead of fifty six.

Local work looks free until you read the right-hand meter. The exchanges it saves come back as
computation, every setting on the list reaches the same place, and which budget a consortium
would rather spend is a fact about its network and its members rather than about the method.

Nothing in parts 3 and 4 has looked at what those exchanges contain. Thirteen exchanges with
four banks is fifty two updates, each of them fourteen numbers computed directly from customer
records. Part 5 asks what can be read out of one.

---
# Part 5 · Can an update reveal a customer?

**The adversary is the coordinator**, and anyone who sees what it sees. Each bank is trusted with
its own records. The coordinator receives every update legitimately: nothing is intercepted and no
cryptography is broken.

Fifty two updates crossed the network in that run, and nobody has yet asked what is in one.

## 1 · What is actually in an update

An update answers "which way should the model move?" For this model it fits on one line: training
on **one customer** changes the thirteen weights and the single bias by

$$\Delta w = e \times x \qquad\qquad \Delta b = e$$

| symbol | what it is |
|:--|:--|
| $x$ | the customer's thirteen numbers |
| $e$ | one number, saying how wrong the model was about that customer |
| $\Delta w$ | the change to the thirteen weights |
| $\Delta b$ | the change to the one bias |

The same $e$ sits in both, because the bias is a weight on a feature that is always 1. So divide
the first by the second and it cancels:

$$\frac{\Delta w}{\Delta b} \;=\; \frac{e \times x}{e} \;=\; x$$

**If an update came from one customer, one division gives that customer back.**

> 📐 Where those two lines come from is the optional section at the end. You do not need it here.

That outcome has a name, and part 6 keeps returning to it.

> **Exact reconstruction.** Recovering a customer's actual feature values from an update, to the
> precision of the arithmetic itself, with no guessing and no auxiliary data. It either happens or
> it does not, which is not the same as learning something about a customer: that is a matter of
> degree, and no defence here claims to prevent it.

Everything turns on the words *one customer*, so the part takes two cases.
**Units.** The thirteen numbers are standardised, so every distance below is on that scale: 1.0
between two customers is about one standard deviation, and 0 is the same person.

## 2 · Case 1, the update came from one customer

Section 1 did the arithmetic. Two things are worth adding before you run it.

**Local training does not save this customer.** A bank that takes five steps sends the sum of five
updates. Each one is some number times the *same* $x$: only the number changes, never the
direction, so the sum is still a number times $x$ and the division still returns it. There is
nothing to mix, so no amount of training mixes it.

**What comes back is standardised.** A recovered age of $-0.04$ is not yet a person, so one step
is left: undo the shift and scale that part 0's dictionary sets for every column.

$$\text{age} = 40 + 10 \times \text{age}_z$$

**Where an attacker gets the 40 and the 10.** Rarely hard, and there is more than one way in.

| route | why it works |
|:--|:--|
| every bank already holds them | all four must scale identically or the shared model means nothing |
| the coordinator holds them | it needs them to deploy the model at all |
| they are often published | population figures, and often suspiciously round |
| two customers held in both forms | two equations, two unknowns, which is what Task 4 asks you to solve |

### 🎯 Task 4 · Recover the customer

The cell below is laid out in these four steps. You fill the three that are marked.

**Algorithm · gradient inversion, one customer.**

| # | what the line does | who writes it |
|--:|:--|--:|
| **1** | the bank trains on one customer and sends `delta_w`, thirteen weight changes, and `delta_b`, one bias change | given |
| **2** | `standardised` ← divide the weight part by the bias part, and the shared factor cancels | **Task 4a** |
| **3** | `scale` ← years per standardised unit: the gap between the two filed ages, over the gap between the same two standardised ages | **Task 4b** |
| | `centre` ← `filed[0] - scale × standardised_pair[0]` | given |
| **4** | `age` ← undo the standardising: the age equation above, read from right to left | **Task 4c** |
| **return** | the standardised customer, the `centre` and `scale` behind them, and the age in years | given |

In [ ]:
victim = books[3]                                  # D Community, the smallest book
mine   = scenario.load_raw()[0].X                  # two customers the attacker holds,
mine_z = books[0].X                                #   in filed units and in the model's
real   = scenario.load_raw()[3].X[0]               # the victim, as the bank filed them

# STEP 1 · what the bank computes, and what leaves the building
one_update = -0.5 * fc.per_record_gradients(fc.init(), victim.X[:1], victim.y[:1])[0]
delta_w = one_update[:fc.N_FEATURES]               # thirteen weight changes
delta_b = one_update[fc.N_FEATURES]                # one bias change

def read_the_customer(delta_w, delta_b, filed, standardised_pair):
    """Turn one bank's update back into the person who produced it."""
    # 🎯 Task 4a — STEP 2 · every one of the thirteen weight changes is the same
    #              shared factor times one of the customer's features, and the bias
    #              change is that factor on its own. Combine the two pieces STEP 1
    #              handed you so the factor cancels and the thirteen features survive.
    standardised = ???
    # 🎯 Task 4b — STEP 3 · you hold two of your own customers in both units: `filed`
    #              is their ages as the bank wrote them down, `standardised_pair` the
    #              same two as the model sees them. One standardised unit is worth
    #              however many years separate that pair, so it is the gap in one
    #              over the gap in the other, both taken between the same two people.
    scale = ???
    centre = filed[0] - scale * standardised_pair[0]
    # 🎯 Task 4c — STEP 4 · undo the standardising: age = centre + scale x age_z, and
    #              age_z is the victim's own standardised age. Age is the first of
    #              the thirteen features, so it is the first entry of `standardised`.
    age = ???
    return standardised, centre, scale, age


standardised, centre, scale, age = read_the_customer(
    delta_w, delta_b, mine[:2, 0], mine_z[:2, 0])
print(f"   recovered, standardised   {standardised[0]:+.4f}")
print(f"   units recovered           age = {centre:.1f} + {scale:.1f} x age_z")
print(f"   recovered age             {age:,.1f} years")
print(f"   the real customer         {real[0]:,.1f} years")

fv.check(
    ("Task 4a", np.allclose(standardised, victim.X[0], atol=1e-8),
     "the division returns the customer exactly as the model sees them",
     "that is not the victim's standardised row — divide delta_w by delta_b"),
    ("Task 4b", np.isclose(scale, (mine[1, 0] - mine[0, 0])
                                  / (mine_z[1, 0] - mine_z[0, 0])),
     f"one standardised unit is {scale:.1f} years",
     "the scale is off — it is the gap in filed years over the gap in standardised units"),
    ("Task 4c", np.isclose(age, real[0], atol=1e-6),
     f"the recovered age matches the real customer to {abs(age - real[0]):.1e} years",
     "the age does not match — read age = centre + scale x standardised[0]"),
)

In [ ]:
#@title 🔓 5.1 — the other twelve measurements, read off the same way { display-mode: "form" }
R = gen.REFERENCE                                  # (centre, scale) per column, from part 0

def read_off(i, label, key, undo):
    z = standardised[i]
    centre, scale = R[key]
    print(f"{label}\n   divide           {z:+.4f}   <- standardised"
          f"\n   undo the units   {undo(centre, scale, z)}"
          f"\n   really filed as  {real[i]:,.2f}\n")

read_off(0, "AGE", "age_years",
         lambda c, s, z: f"{c:g} + {s:g} x ({z:+.4f}) = {c + s * z:,.2f} years")
read_off(1, "INCOME  ·  standardised on the logarithm", "annual_income_k",
         lambda c, s, z: f"exp({c:.4f} + {s:g} x ({z:+.4f})) = {np.exp(c + s * z):,.2f}k")
read_off(3, "DEBT-TO-INCOME", "debt_to_income",
         lambda c, s, z: f"{c:g} + {s:g} x ({z:+.4f}) = {c + s * z:.4f}")
print(f"largest disagreement across all thirteen  "
      f"{np.abs(standardised - victim.X[0]).max():.2e}")

No rule was violated here, because there is no rule. Federated learning controls where records
are stored. It does not control how much information an update carries. And **standardising is
not a privacy mechanism**: it is a fixed shift and scale, reversible by construction, so it
changes the units an attacker reads rather than what they can learn.

**But the consortium's banks do not send one-customer updates.** D Community trains on all 150
of its customers before it sends anything, and that is the update that actually crossed the
network. So the second case is the one that matters, and *not a person* is not the same as
*nothing*.

## 3 · Case 2, the update came from many customers

Same division, one extra ingredient. A batch of $n$ customers sends the sum of their updates, and
dividing that sum no longer returns one person:

$$\frac{\Delta w}{\Delta b} \;=\; c_1 x_1 + c_2 x_2 + \cdots + c_n x_n
\qquad\qquad c_1 + c_2 + \cdots + c_n = 1$$

Each $c_i$ is customer $i$'s share of the answer: their own error, divided by the total of all
$n$ errors. In words: **the division still works. It just returns a weighted average of everybody
in the batch, and the weight on each customer is how wrong the model was about them.**

That one line predicts everything the next two cells measure.

- The shares add to one, so the result sits in the neighbourhood of the group rather than of any
  one person.
- Errors come with both signs, so some shares are **negative**, and the result is not trapped
  between the customers that made it.
- The shares depend on where the model is. At an untrained model every prediction is the same,
  every share is equal, and the blend collapses to the plain average, which is why 5.2
  measures at a part-trained model instead.

5.2 runs the division over thirty two customers and asks who comes back. 5.3 asks what follows
from that: **how much of one customer is still in there?**

In [ ]:
#@title 🔬 5.2 — the same four steps, over thirty two customers { display-mode: "form" }
divide = lambda g: g[:fc.N_FEATURES] / g[fc.N_FEATURES]        # step 2, from case 1
w_mid  = np.random.default_rng(0).normal(0, 0.3, fc.N_PARAMS)  # where a real round is
batch, by = victim.X[:32], victim.y[:32]

# STEP 1 & 2 · the same update, the same division
blend = divide(fc.gradient(w_mid, batch, by))

# STEP 3 · what it is: the shares the formula predicts
c = (fc.predict_proba(w_mid, batch) - by)
c = c / c.sum()
print(f"the formula reproduces the division to   "
      f"{np.abs(blend - (c[:, None] * batch).sum(0)).max():.1e}")
print(f"shares add to {c.sum():.2f}, {int((c < 0).sum())} of the 32 negative, "
      f"largest {np.abs(c).max():.1%}\n")

# STEP 4 · so who is it? nobody in particular, and the group in general
out = books[0].X[np.random.default_rng(1).choice(books[0].n, 200, replace=False)]
print(f"   to the closest of the 32          {np.abs(blend - batch).mean(1).min():.3f}")
print(f"   to the customer we started from   {np.abs(blend - victim.X[0]).mean():.3f}")
print(f"   to the average of the 32          {np.abs(blend - batch.mean(0)).mean():.3f}")
print(f"   to 200 customers at another bank  "
      f"closest {np.abs(blend - out).mean(1).min():.3f}")

In [ ]:
#@title 📊 5.3 — how much of the blend one customer still carries { display-mode: "form" }
SIZES = (1, 2, 4, 8, 16, 32, 64, 150)
med, lo, hi = [], [], []
for B in SIZES:
    share = []
    for seed in range(24):                    # twenty four places a round could be
        w = np.random.default_rng(seed).normal(0, 0.3, fc.N_PARAMS)
        e = fc.predict_proba(w, victim.X[:B]) - victim.y[:B]
        share.append(float(np.abs(e / e.sum()).max()))
    med.append(float(np.median(share))); lo.append(min(share)); hi.append(max(share))

fv.influence_share(SIZES, med, lo, hi,
                   takeaway=scenario.FIGURES["batching_takeaway"])

**How much of one customer survives the averaging?** Read the two lines at the far right, where
the batch is D Community's whole book of 150.

| over 150 customers | one customer's share |
|:--|:--|
| if the update split equally | 0.7% |
| the largest real share, typically | **5.5%**, eight times as much |

Shares are prediction errors, and errors come with both signs, so the total they divide by can
land near zero. That gives the share **no ceiling**: at four of these eight batch sizes, one
customer's share passes 100%.

**Averaging converts the disclosure rather than bounding it.** The exact copy of one person
becomes a description of a group, and how small that group is was never something the bank
promised. Part 6 adds the promise.

**The step count does nothing. The customer count does everything.** Case 1's argument holds for
any number of local steps. Adding a second customer is what breaks the exact copy, and no amount
of extra local work brings it back.

### Would this work on a bigger model?

Partly. The one-line attack is a property of this model; the risk it demonstrates is not.

| | Why the division works, or does not | What an attacker has to do |
|---|---|---|
| **Logistic regression** (this notebook) | One customer's update is a single number times their own row, and adding up steps keeps it that way, so the division survives any amount of local training. | One division. Nothing else. |
| **Larger models** | No single shared factor across the whole update, though the first layer of a plain network keeps a similar shape. | A search: guess an input, compute the update it would produce, compare, adjust. Success depends on architecture, batch size and how much local training happened. |

What generalises is the shape of the risk: how much an update gives away tracks how few records
went into it.

## 4 · Now run the attack yourself

Task 4 did the reconstruction in Python, on an update the cell handed you. This does it on the
wire, on an update you cause a bank to send.

**Heist.** Set one bank to train on a single customer, run a round, then open that bank's
Δw Δb on the coordinator's table and divide. The mission is achieved when what comes back
matches a real customer in that bank's file.

Same frame as before: scroll inside it, and press **Back to the notebook** when the card
appears.

In [ ]:
#@title 🎮 5.4 — mission: the heist (run me, then play) { display-mode: "form" }
fv.mission("heist")

### ✍️ Fill this in — your heist log

**You hand this in.** One line in each ⬜, before reading on.

| What to record | Your answer |
|---|---|
| The dial you turned, and what you set it to | ⬜ |
| The bank you attacked | ⬜ |
| What the division handed back | ⬜ |

One dial decided that. The bank that gave a customer away and the bank that did not were running
the same procedure on the same records; all that differed was how many customers went into a
message.

**Where part 5 leaves us.** Case 1 is exact and needs no cleverness. Case 2 ends the exact copy
and replaces it with a share that is typically eight times an equal split and has no ceiling.
Nothing so far limits what one customer can contribute to what a bank sends, and no rule in the
consortium requires a bank to average over anybody at all.

That is the gap part 6 closes, and a share is the thing it puts a number on.

---
# Part 6 · How do we protect customers?

Part 5 ended on a number with no ceiling. One customer's share of what a bank sends is typically
several times an equal split, and at four of the eight batch sizes some model position pushed a
single share past the whole update.

So ask what a promise would actually require. Not stopping the model from learning population
patterns, which is what it is for. Bounding what any one customer contributes to what gets sent,
and then being able to say by how much.

Three mechanisms are used for this, and they are routinely spoken about as if they were one
thing. They are not.

1. **Batching**, which part 5 already met and already broke.
2. **The cap** a bank puts on its own customers. Section 6.1.
3. **Noise**, which is the only one that buys a promise. Sections 6.2 and 6.3.

Each section answers the same four questions: what it is, who owns it, what it measurably does,
and what it does not do. That last answer is what creates the next one. Section 6.4 then takes
all three into the playground, where you try to stop your own attack.

## Mechanism 1 · Batching, which you have already met

A bank that computes its update over 150 customers instead of one destroys the exact
reconstruction. Part 5 measured what that buys and what it does not: the largest share falls
from the whole update to about 5.5%, which is eight times an equal split, and at some model
positions it goes back above 100%.

**Owner: the bank, by choice.** Nobody enforces it, and a bank that sends a single record update
has broken no rule.

**What it does not do:** put a number on anything. So the next mechanism puts a number in.

## Mechanism 2 · The cap a bank puts on its own customers

Before a bank adds up its customers' contributions, it can cut each one down to a fixed maximum
size. This is **customer gradient clipping**, and it is one rule:

$$\text{if } s_i > C \quad\Longrightarrow\quad g_i \;\rightarrow\; g_i \times \frac{C}{s_i}$$

| symbol | what it is |
|:--|:--|
| $g_i$ | customer $i$'s contribution: the fourteen numbers they would add to the update |
| $s_i$ | how big that contribution is, as one number |
| $C$ | the cap, the most any one customer is allowed to contribute |

Oversized contributions shrink to exactly the cap, smaller ones are left alone. Only the size
ever changes, never the direction. **Owner: the bank**, inside its own building, where nobody
outside can check it.

**Where to put the cap** should come from the data, so the cell below takes the middle of all
2,450 contributions in the consortium: too high and it shortens nobody, too low and it throws the
signal away. It also sizes the next mechanism's noise, which is $z \times C$ wide.

Watch for two answers pointing opposite ways. **No customer can exceed the cap**, by
construction, so there is now a worst case where there was none. **And no cap stops part 5's
reconstruction**, because clipping multiplies the whole update by one number, and that number
cancels in the division. The cap bounds a contribution without hiding it, which is what the noise
is for.

In [ ]:
#@title ✂️ 6.1 — a dozen customers, a cap, and what each of them does
per_customer = fc.per_record_gradients(fc.init(), victim.X, victim.y)
sizes    = np.linalg.norm(per_customer, axis=1)     # one number per customer
everyone = fc.contribution_sizes(fc.init(), books)  # all 2,450 in the consortium
C = float(np.median(everyone))                      # the cap: the middle one

show = np.argsort(sizes)[::-1][np.linspace(0, victim.n - 1, 12).astype(int)]
print(f"{'customer':>12}{'contributes':>14}{'after the cap':>16}")
for i in show:
    print(f"{'#' + str(i):>12}{sizes[i]:>14.2f}{min(sizes[i], C):>16.2f}"
          f"{'   shortened' if sizes[i] > C else ''}")
print(f"\nthe cap is {C:.2f}, the middle of all {len(everyone):,} contributions")

display(fv.cap_explorer([f"#{i}" for i in show], sizes[show], population=everyone))

print("does the cap stop part 5's reconstruction?")
for limit in (1.0, 0.1, 0.001):
    g = one_update * min(1.0, limit / np.linalg.norm(one_update))
    print(f"   capped at {limit:<7} the recovered customer still differs by "
          f"{np.abs(divide(g) - victim.X[0]).max():.2e}")

## Mechanism 3 · Noise, which is what actually buys a promise

Adding an unpredictable number to the total is the only step here that makes a customer's
presence genuinely uncertain. It is one line too:

$$\text{what the bank sends} \;=\; g_1 + g_2 + \cdots + g_n \;+\; \text{noise}$$

| symbol | what it is |
|:--|:--|
| $g_1 \ldots g_n$ | the capped contributions, from mechanism 2 |
| $z$ | the noise multiplier, the dial the bank turns |
| $z \times C$ | the width of the bell curve the noise is drawn from |

In words: add up whatever survived the cap, then add one random number to each of the fourteen
entries. Which is why this mechanism means nothing without the previous one: you cannot choose a
width until you know the most a single customer could have contributed. **Owner: the bank again**,
since clipping and noise both happen before anything is sent.

**What it does not do:** bind anyone who declines to apply it. Like the other two it runs inside
a bank, so it is a promise rather than a control.

In [ ]:
#@title 🔇 6.2 — what noise does to the same attack
zs = (0.01, 0.05, 0.1, 0.5, 1.0)
rng = np.random.default_rng(7)
errs = []
for z in zs:
    trials = []
    for _ in range(300):
        noisy = one_update + rng.normal(0, z, fc.N_PARAMS)
        trials.append(np.abs(divide(noisy) - victim.X[0]).mean())
    errs.append(float(np.median(trials)))
fv.recovery_vs_noise(zs, errs)

### 🎯 Task 5 · Mechanisms 1, 2 and 3, in one function

This is **DP-SGD**, Differentially Private Stochastic Gradient Descent. Larger $z$ buys a
stronger guarantee and a worse model, and that bound is what turns the mechanism into a number
with a name.

**Algorithm · DP-SGD, one private step.**

| # | what the line does | who writes it |
|--:|:--|--:|
| **input** | `w` the model · `C` the cap from mechanism 2 · `z` the noise multiplier · one lot of customers | |
| **1** | `per_customer` ← one gradient row per customer | given |
| **2** | `norms` ← how long each of those rows is | **Task 5a** |
| **3** | `factors` ← `min(1, C / norms)`, so never above 1 | given |
| **4** | `clipped` ← each row scaled by its own factor | **Task 5b** |
| **5** | `noisy` ← the clipped rows added up, then one Gaussian draw of width `z × C` | **Task 5c** |
| **return** | `w` stepped against `noisy`, by `lr / lot` | given |

> ### $\varepsilon$
>
> **The privacy number, and it is a bound on a ratio.** Take a bank's book, and the same book
> with one customer taken out. Whatever the coordinator ends up seeing, it is at most
> $e^{\varepsilon}$ times as likely to have come from the one as from the other.
>
> **Smaller is stronger.** At $\varepsilon = 0$ the two are indistinguishable, and nothing sent
> could betray whether that customer was there. It is not a promise that nothing is learned about
> a bank's customers, only that little of it belongs to any single one of them.
>
> **Read $e^{\varepsilon}$, not $\varepsilon$.** The gap from 0.5 to 4.7 is not nine times
> weaker, it is $e^{4.2}$, about **67 times**. It is applied inside the bank, and covers all 65
> noisy steps together.

> 📐 The accountant is `fc.epsilon`, which turns $C$, $z$, the sampling and a small $\delta$
> into these numbers. Appendix item 4 opens it up.

In [ ]:
def dp_step(w, X, y, C=C, z=4.0, lot=64, lr=0.5, rng=None):
    rng = rng or np.random.default_rng(0)
    per_customer = fc.per_record_gradients(w, X, y)    # one row per customer

    # 🎯 Task 5a — MEASURE · how big is each customer's contribution? one number per row,
    #              kept as a column so it divides into the rows cleanly
    norms   = ???
    factors = np.minimum(1.0, C / (norms + 1e-12))
    # 🎯 Task 5b — CAP · nobody counts for more than C: scale each row by its own factor
    clipped = ???
    # 🎯 Task 5c — DISTURB · add up the capped rows, then add noise of width z * C, once
    noisy   = ???
    return w - lr * noisy / lot


_w = dp_step(fc.init(), books[3].X, books[3].y)
print(f"one private step taken, model moved by {np.abs(_w - fc.init()).max():.4f}")

_Xd, _yd = books[3].X, books[3].y
_per  = fc.per_record_gradients(fc.init(), _Xd, _yd)
_open = dp_step(fc.init(), _Xd, _yd, C=1e9, z=0.0)         # cap so wide nothing is cut
_tight = dp_step(fc.init(), _Xd, _yd, C=1e-9, z=0.0)       # cap so tight everything is
_quiet = dp_step(fc.init(), _Xd, _yd, z=0.0, rng=np.random.default_rng(11))
_loud  = dp_step(fc.init(), _Xd, _yd, z=4.0, rng=np.random.default_rng(11))
_want  = -0.5 * np.random.default_rng(11).normal(0, 4.0 * C, fc.N_PARAMS) / 64
fv.check(
    ("Task 5a", np.allclose(_tight, fc.init(), atol=1e-9),
     "a tight cap holds every customer down, so the model barely moves",
     "a tiny cap still moved the model — norms must be each row's own length, "
     "np.linalg.norm(..., axis=1, keepdims=True)"),
    ("Task 5b", np.allclose(_open, fc.init() - 0.5 * _per.sum(axis=0) / 64),
     "with the cap wide open the step is the plain sum of the contributions",
     "with nothing to cut, clipped should be per_customer * factors and change nothing"),
    ("Task 5c", np.allclose(_loud - _quiet, _want),
     "and the noise added is one draw per parameter, of width z x C",
     "the noise is not one draw of width z * C added to the summed rows"),
)

### The same procedure, four different promises

The cell below runs the whole federation at eight settings of $z$, and reports what the shared
model scores and what $\varepsilon$ each bank can claim. The four will not agree, and the reason
is sampling. Each step uses a small random sample of a bank's customers, and a customer left out
of a step has no influence on it. Being hidden in a crowd is part of the protection, and the
crowd is the size of the bank. A Prime Metro draws 64 from 1,250, so a customer appears in about
one step in twenty. D Community draws 64 from 150, so most of its customers are in most steps
and there is almost no crowd to hide in.

**The consortium's promise is the weakest of its four, not their average.** Every customer here
belongs to exactly one bank, so an average would be a promise nobody could keep. The bank with
the most to gain from the federation is the one whose customers are protected least.

In [ ]:
#@title 🔐 6.3 — what the noise dial costs, and what it buys { display-mode: "form" }
fed13 = fc.run(list(books), E=5, rounds=13, lr=0.5)["w"]   # the same run, with no privacy at all

ZS = (1.0, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0, 16.0)
sweep_auc, sweep_worst = [], []
per_bank = {b.name: [] for b in books}
for z in ZS:
    p = fc.Privacy(C=C, z=z, lot=scenario.FIGURES["lot"])
    ws = [fc.run(list(books), E=5, rounds=13, lr=0.5, privacy=p, seed=s)["w"]
          for s in range(5)]
    e = fc.epsilon_per_site(books, p, rounds=13, E=5)
    sweep_auc.append(float(np.mean([fc.auc(w, Xc, yc) for w in ws])))
    sweep_worst.append(float(np.mean([min(fc.auc(w, c.X, c.y) for c in cohort) for w in ws])))
    for name, v in e.items():
        per_bank[name].append(v)

fv.privacy_sweep(ZS, sweep_auc, sweep_worst, per_bank, baseline=fc.auc(fed13, Xc, yc))

## Mission · now stop yourself

The last mission is the other half of the heist: same console, same attack, and this time your
job is to make it fail.

**Now stop yourself.** Break the exact match **and** keep the shared model at 0.84 or better.
Batching, the cap and the noise dial are all live, and the mission is achieved only when both
hold at once.

The second condition is the whole point. Any one of the three dials stops the attack if you
turn it far enough. What part 6 has been measuring is what that costs everybody else.

In [ ]:
#@title 🎮 6.4 — mission: now stop yourself (run me, then play) { display-mode: "form" }
fv.mission("privacy")

### ✍️ Fill this in — your defence log

**You hand this in.** One line in each ⬜. The mission needs both halves at once, a broken
attack *and* a useful model, so record both.

| What to record | Your answer |
|---|---|
| The batch size that broke the attack | ⬜ |
| The cap you used | ⬜ |
| The noise multiplier z you used | ⬜ |
| The shared model's score at those settings | ⬜ |
| Which of the three dials did the most, and what it cost | ⬜ |

You found a setting by trying it. A consortium has to choose one in advance, write it into an
agreement and live with it, and the ε badge on each bank is the number that agreement would
have to quote.

**Where part 6 leaves us.** Three mechanisms, and the useful way to hold them apart is by what
each one gives you. Batching converts the disclosure without bounding it. Clipping bounds one
customer's contribution but puts no number on the result. Noise is the only one that buys a
promise, and the cap is what sizes it.

All three run inside a bank, on records nobody else sees, so all three are promises rather than
controls. **What the coordinator gets is the submitted update and nothing else**, which says what
a bank sent but not whether it batched, clipped or added noise first. None of them settles what
6.3's noise sweep showed either: buying an arbitrary customer a strong promise costs everybody
accuracy.

That is a decision rather than a measurement, and it is the last thing the required parts
measure. Every number in them came from fourteen parameters and four participants.

---
# What you built

Six ideas from the required parts, and what each one turned out to mean once it was measured.
The seventh row is there if you go on to the optional part 7.

| Part | The idea | What it turned out to mean |
|---|---|---|
| 1 | baselines, and choosing a metric | accuracy flatters a rare outcome; AUC and the worst bank do not |
| 2 | data that is not identically distributed | uneven outcomes cost, uneven customers do not |
| 3 | FedSGD, and size-weighted aggregation | the $n_k/N$ weights are derived from pooling, never chosen |
| 4 | FedAvg, and the three budgets | local work buys exchanges with computation; it moves the cost rather than removing it |
| 5 | gradient inversion | one record gives the person back; many give the group, with no ceiling |
| 6 | clipping, noise, and $\varepsilon$ | only noise buys a promise, and every promise is made inside a bank |
| 7 *(optional)* | scaling | parameters, participants and payload leave the procedure alone and multiply the bill |

The one sentence under all of it: **federated learning decides where records are stored, and
nothing else.** Quality, cost and disclosure each have to be measured separately, and every
number above came from these four banks rather than from the idea.

---
### ✍️ Fill this in — closing reflection

**You hand this in.** **At most two short sentences in each ⬜.** No more than that is wanted.
Every answer is somewhere in this notebook and the part that carries it is named, so none of
these needs guessing.

| Question | Your answer, in two sentences at most |
|---|---|
| **1.** Part 3 argues that federated training is not an approximation of pooled training but the same thing computed in four places. What has to hold for that to be exactly true, and which one of those conditions does part 4 break on purpose? *(part 3)* | ⬜ |
| **2.** Part 6 calls all three mechanisms promises rather than controls. What do the three have in common that makes them impossible to check from outside a bank, and what is the one thing the coordinator does get to see? *(parts 5 and 6)* | ⬜ |
| **3.** D Community High-Touch gains the most from the federation and carries its weakest privacy number. Both facts trace back to the same property of that bank. What is it, and why must the consortium quote 4.7 rather than the average of the four? *(parts 1 and 6)* | ⬜ |
| **4.** Part 4 reaches the same one per cent target in thirteen exchanges instead of fifty six. What does that saving cost, and which of 4.2's two meters is where the cost shows? *(part 4)* | ⬜ |

**That is the hand-in.** With the tables above filled and the five required task cells run, the
notebook is complete and you can stop here.

**Optional from here.** What follows asks the same questions at sizes these four banks will never
reach, and derives two things the notebook used without proving. It is worth the time if the
consortium you have in mind is larger than four members.

---
# Part 7 · What happens when the system gets bigger? *(optional)*

**Optional.** Everything the consortium has to decide is settled by part 6, and nothing earlier
needs this part. It prices the same procedure at sizes these four banks will never reach.

Everything so far ran on one shape of system: four participants, fourteen parameters, everybody
present at every exchange. Deployed systems are not that shape, and they grow along three axes
at once.

| | The axis | What grows | What it does to the bill | Where |
|---|---|---|---|---|
| **1** | **Model size** | parameters per message | every message gets larger | 7.1, 7.2 |
| **2** | **Federation size** | participants | there are more messages | 7.3, 7.4 |
| **3** | **Payload** | what each message actually carries | the only one you can shrink | 7.5, 7.6 |

The procedure survives all three untouched. Aggregation, local steps, clipping, noise and the
accountant never ask what produced a gradient or how many produced it. The bill does not
survive, and the rest of this part prices it, one axis at a time.

## 1 · Model size, and why every message gets larger

The model has been fourteen numbers since part 0. The two cells below run the same federation on
a wider one, changing nothing but the width, then price the message that width produces.

**Two honest notes on the wider run.** It scores slightly worse here, because the extra features
carry no real signal and a bigger model is not automatically a better one. And a real neural
network would change more than the width: the optimisation gets harder, clipping every customer
separately gets expensive, and part 5's one line division becomes a search. The accountant stays
valid as long as the sampling, clipping and noise stay as described.

**What one exchange costs.** Both directions, every time. The coordinator sends the whole model
down to each of the four banks, and each bank sends a whole update back up, at eight bytes a
number. Fourteen parameters is therefore 2 × 4 × 14 × 8 = 896 bytes an exchange. That arithmetic
does not care what the numbers mean, so it prices every model in 7.2.

In [ ]:
#@title 🔁 7.1 — the same federation, a wider model { display-mode: "form" }
def expand(X):
    return np.c_[X, X[:, :6] ** 2, X[:, 0:1] * X[:, 3:4]]

big_books = fc.Federation([fc.Site(b.name, expand(b.X), b.y) for b in books])
big_cohort = fc.Federation([fc.Site(c.name, expand(c.X), c.y) for c in cohort])
fc.configure(expand(books[0].X).shape[1])

big_w = fc.run(list(big_books), E=5, rounds=13, lr=0.5)["w"]
print(f"model width      {fc.N_PARAMS} parameters")
print(f"same fc.run      AUC {fc.auc(big_w, *big_cohort.pooled()):.4f}")
print(f"bytes exchanged  {2 * len(books) * fc.N_PARAMS * 8:,} per exchange")
_ = fc.configure(scenario.N_FEATURES)   # back to fourteen for the rest of the notebook

In [ ]:
#@title 📶 7.2 — the same message, with more numbers in it { display-mode: "form" }
def exchange_bytes(params, banks=4):        # both directions, every bank, 8 bytes a number
    return 2 * banks * params * 8

fv.model_widths([
    ("this project — 14 parameters", 14, exchange_bytes(14), None),
    ("a small neural network — 1,900", 1900, exchange_bytes(1900), None),
    ("ResNet-18 — 11.7 million", 11.7e6, exchange_bytes(11.7e6), None),
    ("a 7-billion-parameter model", 7e9, exchange_bytes(7e9), fv.BAD),
])

## 2 · Federation size, and why there are more messages

Four hundred and forty-eight gigabytes an exchange is the wall, and the two axes that follow are
the ways through it. Four is not the interesting number of participants either.

| | who they are | how many | who you can wait for |
|:--|:--|:--|:--|
| **cross-silo** | named institutions, each with a large book and a signature on a contract | a handful | all of them |
| **cross-device** | phones, branches, terminals, each holding very little | hundreds or thousands | almost none |

The consortium is the first row. The procedure is identical in both; the assumptions are not, and
one breaks immediately: you can no longer wait for everybody. So you stop asking. **Client
sampling** draws a small subset each exchange, and it is how a system of this size is designed to
run, not a degradation to be tolerated.

The two cells below try it at four, five hundred and a thousand participants, with two per cent
of the large ones speaking. All three get the same twenty exchanges, or the comparison would be
about rounds rather than about sampling. The model lands in much the same place either way: a
round needs a representative sample of the population, not a complete census of it. So **the
participant count in the bill is the number sampled, not the number enrolled**.

In [ ]:
#@title 🛰️ 7.3 — the same procedure, a thousand participants { display-mode: "form" }
# a hypothetical: the same four populations, spread thinly over many small holders
wide = gen.generate(sizes={b["name"]: 2500 for b in gen.BANKS}, seed=777)
Xw = np.vstack([Xz for _, _, _, Xz, _, _ in wide])
yw = np.concatenate([y for _, _, _, _, y, _ in wide])

def fleet(n_devices, seed=0):
    idx = np.random.default_rng(seed).permutation(len(yw))
    return [fc.Site(f"d{k}", Xw[p], yw[p])
            for k, p in enumerate(np.array_split(idx, n_devices))]

ROUNDS = 20                        # the same budget for all three, or this proves nothing
print(f"{'participants':>13}{'records each':>14}{'spoken to':>11}{'AUC':>8}")
FLEET = []
for n, frac in ((4, 1.00), (500, 0.02), (1000, 0.02)):
    sites = list(books) if n == 4 else fleet(n)
    k = max(1, int(round(n * frac)))
    w = fc.run(sites, E=5, rounds=ROUNDS, lr=0.5, sample=k)
    auc = fc.auc(w["w"], Xc, yc)
    per = fc.N_PARAMS * 8 * 2
    FLEET.append((f"{n:,} participants", n, k, per * n, per * k, auc))
    print(f"{n:>13,}{int(np.mean([s.n for s in sites])):>14}{k:>11}{auc:>8.3f}")

In [ ]:
#@title 📡 7.4 — who speaks, and what that saves { display-mode: "form" }
fv.federation_widths(FLEET)

## 3 · Payload, the only axis you can shrink

Send less of each update. Both levers so far change *how many* numbers cross the wire. The
third changes *how much each number costs*, and it is the one that survives into the largest
systems.

Two standard moves, both a few lines:

| | what it sends | what it throws away |
|:--|:--|:--|
| **Sparsification**, top-$k$ | the largest few changes, and zeros for the rest | everything below the cut, on the argument that most of an update is close to noise anyway |
| **Quantisation**, signSGD | the direction each number moved, and one shared magnitude | every individual magnitude, at one bit a number |

Write the first of them, since the second is given to you. Section 7.5 then draws all three forms
side by side on one real update, and what the smaller ones cost in accuracy is the last thing this
part measures.

### 🎯 Task 6 · Send only the largest changes

`one_bit` is written for you: one magnitude shared by the whole update, times the direction of
each number in it. Write the other one, which keeps the `k` biggest changes and zeros the rest.

In [ ]:
def one_bit(delta):                      # one size for the whole update, each direction kept
    return np.abs(delta).mean() * np.sign(delta)

def sparsify(delta, k):
    out = np.zeros_like(delta)
    # 🎯 Task 6 — where the k biggest changes are, size only, direction ignored.
    #             np.argsort hands back positions, smallest first, so take from the end
    keep = ???
    out[keep] = delta[keep]
    return out


_d = np.array([0.30, -0.06, 0.03])
print("an update          ", _d)
print("top 1 of 3         ", sparsify(_d, 1))
print("one bit per number ", np.round(one_bit(_d), 3))

fv.check(
    ("Task 6", np.allclose(sparsify(_d, 1), [0.30, 0.0, 0.0])
                and np.allclose(sparsify(_d, 2), [0.30, -0.06, 0.0]),
     "the largest changes survive whichever way they point",
     "top-2 of [0.30, -0.06, 0.03] should keep 0.30 and -0.06 — rank by "
     "np.abs(delta), not by the value"),
)

In [ ]:
#@title 🗜️ 7.5 — the three forms one update can travel in { display-mode: "form" }
P = fc.N_PARAMS

# one real update to draw the three forms with: A Prime's first five local steps
_w0 = fc.init()
_local = _w0.copy()
for _ in range(5):
    _local = _local - 0.5 * fc.gradient(_local, books[0].X, books[0].y)

fv.wire_formats(_local - _w0, 4, (P * 8, 4 * 8 + 4, P / 8 + 8))

### And what does the payload dial cost?

Bytes are only half the question. Everything so far priced what each axis does to the bill, and
none of it says what the smaller message costs in quality. The last figure measures that. It
takes the payload dial, the one a consortium actually chooses, and tries each setting against the
other two axes: more local work along the first panel, more participants along the second.

The four lines start far apart on the left of each panel and close as you move right. Three
things to read off them:

| | what it costs | why |
|:--|:--|:--|
| **one bit**, one local step | about two hundredths of AUC, more than everything else in this part combined | it keeps an update's direction and throws away its size, and one small noisy step is mostly size |
| **one bit**, more of everything | nothing at fifty local steps, and the second panel stops improving at about ten participants | ten independent signs already recover the direction, and the next four hundred and ninety add almost nothing |
| **top 4 of 14** | close to nothing, across both panels | the largest four changes are most of what the update was going to say |

So compression is not a fixed discount. It is a bargain whose price depends on the rest of the
configuration, and a consortium that already does heavy local work can afford the most aggressive
setting on the list. One caveat, and it is part 3's again: all of this is AUC, and a setting that
looks free here has not been shown to be free on loss.

In [ ]:
#@title 🗜️ 7.6 — when compression is free, and when it is not { display-mode: "form" }
def squeeze(delta, how):
    if how == "full": return delta
    if how == "sign": return np.abs(delta).mean() * np.sign(delta)
    keep = 4 if how == "top4" else 2
    out = np.zeros_like(delta)
    idx = np.argsort(np.abs(delta))[-keep:]
    out[idx] = delta[idx]
    return out

FLEETS = {4: list(books)}
for n in (500, 1000):
    order = np.random.default_rng(0).permutation(len(yw))
    FLEETS[n] = [fc.Site(f"d{k}", Xw[p], yw[p])
                 for k, p in enumerate(np.array_split(order, n))]

def measure(n, speaking, E, how="full", rounds=20, seed=0):
    rng = np.random.default_rng(seed)
    w, sites = fc.init(), FLEETS[n]
    for _ in range(rounds):
        here = sites if speaking >= len(sites) else [
            sites[i] for i in rng.choice(len(sites), speaking, replace=False)]
        ups, sizes = [], []
        for site in here:
            local = w.copy()
            for _ in range(E):
                local = local - 0.5 * fc.gradient(local, site.X, site.y)
            ups.append(squeeze(local - w, how)); sizes.append(site.n)
        w = w + fc.aggregate_weighted(ups, np.asarray(sizes, float))
    return fc.auc(w, Xc, yc)

def average(*a, seeds=3, **kw):
    return float(np.mean([measure(*a, seed=s, **kw) for s in range(seeds)]))

POOLED_AUC = fc.auc(fit_local(Xp, yp, steps=20000), Xc, yc)

SCHEMES = (("everything", "full"), ("top 4 of 14", "top4"),
           ("top 2 of 14", "top2"), ("one bit", "sign"))

fv.compression_conditions(
    (["1", "5", "20", "50"],
     [(s, [average(4, 4, E, how=h) for E in (1, 5, 20, 50)]) for s, h in SCHEMES]),
    (["4 banks", "5", "10", "50", "250", "500"],
     [(s, [average(4, 4, 5, how=h)]
          + [average(500, k, 5, how=h) for k in (5, 10, 50, 250, 500)])
      for s, h in SCHEMES]),
    reference=POOLED_AUC)

### What this means for these four banks, and for anything larger

Their bill is 896 bytes per exchange, so the whole run costs about 12 kB. Communication is not
a constraint for them and will not become one at this model size. None of the three levers is
worth pulling.

Every one of them matters the moment the consortium wants something larger. At seven billion
parameters those same thirteen exchanges move about 5.8 terabytes. Sending one bit per number
instead of eight takes that to about 90 gigabytes, and sampling a fraction of the participants
takes off however large the federation grew. Multiply the axes and a bill that was impossible
becomes one somebody can sign.

**Where part 7 leaves us.** The procedure they learned in part 3 is unchanged through all of it.
The contract is not: every number in this notebook was measured on four banks and fourteen
parameters, and an agreement that covers those does not automatically cover anything built on
top of them.

---
### Optional · where the update comes from

Part 5 used two lines without proving them: training on one customer changes the weights by
$\Delta w = e \times x$ and the bias by $\Delta b = e$. Here is where they come from. Nothing
later depends on this, so read it only if you want to.

**The setup.** The model adds up the customer's numbers, $z = w \cdot x + b$, and turns that
score into a probability with the sigmoid, $p = \sigma(z)$. Training makes $p$ move towards $y$,
which is 1 if the customer defaulted and 0 if they did not.

**The one convenient fact.** The sigmoid and the loss have derivatives that cancel each other
almost entirely, and what survives is as simple as it could be:

$$\frac{\partial L}{\partial z} \;=\; p - y$$

Predicted minus actual. That is the error, and it is one number per customer.

**The rest is bookkeeping.** The score $z = w \cdot x + b$ changes with each weight $w_j$ at rate
$x_j$, and with the bias at rate 1, because the bias is a weight on a feature that is always 1.
So the error is passed on to the weights multiplied by $x$, and to the bias multiplied by 1:

$$\frac{\partial L}{\partial w} = (p - y)\,x \qquad\qquad \frac{\partial L}{\partial b} = (p - y)$$

A training step of size $\eta$ subtracts $\eta$ times each of those, which gives part 5's two
lines with $e = -\eta\,(p - y)$:

$$\Delta w = -\eta\,(p-y)\,x = e \times x \qquad\qquad \Delta b = -\eta\,(p-y) = e$$

Both carry the same $e$. That is the whole reason the division works, and the reason an attacker
needs neither the learning rate nor the model's prediction: they cancel.

**Recovering the units.** Part 5 also turned a standardised value back into filed units. If a
column is standardised as $x_z = (x - \mu)/\sigma$, two records held in both forms give two
equations in the two unknowns:

$$\sigma = \frac{x^{(1)} - x^{(2)}}{x_z^{(1)} - x_z^{(2)}}
\qquad\qquad \mu = x^{(1)} - \sigma\,x_z^{(1)}$$

---
### Appendix, optional

Nothing in the notebook depends on any of these. Each opens a separate line of thought.

1. **A twenty bank simulation.** Does the small bank advantage hold with more members?
2. **Model poisoning.** A member that sends deliberately wrong updates, and the point at which
   identity control stops helping.
3. **Gradient recovery for a neural network.** Part 5's division becomes an optimisation.
4. **Inside the accountant.** Rényi orders, and why $\varepsilon$ moves the way it does.
5. **Regenerate the federation.** Change the seed, the bank sizes or the risk rule and re-run
   parts 1, 2 and 4. Which conclusions survive? The small bank's gain does, on ten draws out of
   ten. Some do not.

### Where this goes next

Everything in this notebook has a name, and the names are what to search for.

**Training.** FedAvg is the method part 4 measured. It assumes the coordinator waits for
everyone, and *asynchronous* federated learning drops that. *Personalised* federated learning
drops the idea of one shared model entirely, which is the honest answer when participants really
do disagree about the rule.

**Privacy.** DP-SGD is the mechanism in part 6, and Rényi accounting is how its budget is added
up across rounds. *Gradient inversion* is the general name for part 5's attack once it stops
being a division and becomes a search. *Secure aggregation* is the cryptography, and *local
differential privacy* is what you use when you do not trust the coordinator at all.

**Scale.** *Parameter-efficient fine-tuning* is how a large model is made cheap enough to send
at all, and *federated fine-tuning* of language models is where most of the current work is. It
is the same three budgets with different numbers.

The open problems are the ones this notebook kept running into: who audits a promise a
participant makes about its own data, and what a consortium does when its members are not
equally protected by the same setting.